# MedSeg 关键功能演示\n
\n
本 Notebook 演示本仓库的两条关键链路：\n
\n
1. **RAG Top-k 检索**：输入查询图片路径，返回相似样例 `[(img_path, mask_path, distance), ...]`\n
2. **SAM3 memory bank 分割**：将 Top-k 样例（带专家 mask）写入 memory bank，并在锁定 memory bank 后对目标帧进行分割，输出二值掩码 PNG。\n
\n
## 重要说明\n
- 本仓库开发约束中要求**开发过程中不运行项目代码**、**不配置环境**。因此本 Notebook 默认不执行任何单元格；你可以在自己的环境中按需运行。\n
- 运行 SAM3 推理需要 GPU/CUDA 以及权重/依赖；如未准备好，可仅运行“Top-k 解析/检查”等轻量单元格。\n

## 0. 准备：路径与参数\n
\n
请把下面路径替换成你本地真实的文件路径：\n
- `QUERY_IMAGE_PATH`：待测图片（目标分割）\n
- `SAM3_CKPT_PATH`：SAM3 模型权重（可选；不填则交给 SAM3 builder 自行处理）\n
- `RAG_INDEX_DIR`：RAG 索引目录（可选；不填则用 rag/config.py 默认）\n

In [ ]:
from pathlib import Path

QUERY_IMAGE_PATH = "/abs/path/to/query.png"
SAM3_CKPT_PATH = "/abs/path/to/sam3_checkpoint.pt"  # 可选
RAG_INDEX_DIR = "/abs/path/to/rag_index"  # 可选

TOP_K = 5
OUTPUT_PROB_THRESH = 0.5
LOCK_MEMORY = True

assert Path(QUERY_IMAGE_PATH).exists(), f"query image not found: {QUERY_IMAGE_PATH}"

## 1. RAG：检索 Top-k\n
\n
RAG 系统入口是 `rag.rag_system.RAGSystem`。核心方法：\n
\n
- `search(query_image, k) -> [(img_path, mask_path, distance), ...]`\n
\n
`mask_path` 可能为空字符串（表示该图没有配置 mask）。下游分割会对缺失 mask 的条目做跳过/降级处理。\n

In [ ]:
from rag.rag_system import RAGSystem

rag_kwargs = {}
if RAG_INDEX_DIR:
    rag_kwargs["index_dir"] = RAG_INDEX_DIR

rag = RAGSystem(**rag_kwargs)
topk = rag.search(query_image=QUERY_IMAGE_PATH, k=TOP_K)

topk[:3]

## 2. Top-k 解析与可用性检查（不跑模型）\n
\n
`seg_pipeline` 支持直接消费 RAG 的返回结构，并提供输入解析与过滤：\n
- 解析：`normalize_retrieval_topk`\n
- 过滤：`filter_usable_retrieval_items`（要求 image/mask 路径都存在）\n

In [ ]:
from seg_pipeline.protocol import normalize_retrieval_topk, filter_usable_retrieval_items

normalized_items, parse_skipped = normalize_retrieval_topk(topk)
usable_items, skipped = filter_usable_retrieval_items(normalized_items)

print("input_count:", len(normalized_items))
print("usable_count:", len(usable_items))
print("parse_skipped:", parse_skipped[:2])
print("skipped:", skipped[:2])

usable_items[:2]

## 3. SAM3 memory bank 分割（关键链路）\n
\n
### 机制简述\n
为启用 SAM3 的记忆注意力机制，`seg_pipeline` 会把 Top-k 样例帧 + 目标帧物化为一个“合成视频帧序列”（`0.jpg..N.jpg`）。\n
\n
执行流程：\n
1. 对每个 Top-k 样例帧调用 `add_new_mask` 写入专家掩码（作为该帧的 mask prompt）\n
2. `propagate_in_video_preflight(run_mem_encoder=True)`：将条件帧编码写入 memory bank\n
3. 对目标帧传播/推理：当 `LOCK_MEMORY=True` 时，目标帧阶段强制 `run_mem_encoder=False`，实现 **memory bank 锁定（只读）**\n
\n
注意：这一步会加载 SAM3 模型并执行推理。\n

In [ ]:
from seg_pipeline import segment_image
from seg_pipeline.sam3_memory_segmenter import Sam3BuildConfig

sam3_cfg = Sam3BuildConfig(
    version="sam3.1",  # "sam3" 或 "sam3.1"
    checkpoint_path=SAM3_CKPT_PATH or None,
    bpe_path=None,
    compile=False,
    warm_up=False,
    async_loading_frames=True,
)

result = segment_image(
    query_image_path=QUERY_IMAGE_PATH,
    retrieval_topk=topk,
    sam3_build_config=sam3_cfg,
    output_prob_thresh=OUTPUT_PROB_THRESH,
    lock_memory=LOCK_MEMORY,
)

result.output_mask_path, result.meta_path

## 4. 查看输出（mask + meta）\n
\n
- mask：PNG 单通道 8-bit（0/255）\n
- meta：JSON，包含使用的 Top-k、跳过原因、是否锁定等信息\n

In [ ]:
import json
from PIL import Image
from IPython.display import display

mask_img = Image.open(result.output_mask_path)
display(mask_img)

with open(result.meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta.keys()

## 5. 另一种用法：只传 `k`，由流水线内部调用 RAG\n
\n
当你不想手动先调 RAG 时，可以只传 `k`，并用 `rag_kwargs` 指定索引目录等参数（可选）。\n

In [ ]:
from seg_pipeline import segment_image
from seg_pipeline.sam3_memory_segmenter import Sam3BuildConfig

sam3_cfg = Sam3BuildConfig(version="sam3.1", checkpoint_path=SAM3_CKPT_PATH or None)

rag_kwargs = {}
if RAG_INDEX_DIR:
    rag_kwargs["index_dir"] = RAG_INDEX_DIR

result2 = segment_image(
    query_image_path=QUERY_IMAGE_PATH,
    k=TOP_K,
    rag_kwargs=rag_kwargs,
    sam3_build_config=sam3_cfg,
    output_prob_thresh=OUTPUT_PROB_THRESH,
    lock_memory=True,
)

result2.output_mask_path

## 6. 常见问题\n
\n
1. **Top-k 可用条目为 0**\n
   - 说明 Top-k 的 `mask_path` 缺失/为空/文件不存在；流水线会跳过这些条目。\n
   - 你需要确保检索库里每条样例图片都有对应的专家 mask，并且路径可访问。\n
\n
2. **SAM3 权重/依赖问题**\n
   - `Sam3BuildConfig.checkpoint_path` 可以指向本地权重；也可以留空让上游 builder 自行处理。\n
   - 若在你的环境中运行报依赖缺失，请按仓库 `requirements.txt` / `rag/requirements.txt` 补齐依赖。\n
\n
3. **为什么要锁定 memory bank**\n
   - 目标帧如果继续写入 memory，会把“检索样例记忆”稀释/覆盖，导致结果不稳定。\n
   - 因此默认建议 `lock_memory=True`（目标帧阶段 `run_mem_encoder=False`）。\n